# 04 Run the six test tickets
Run the complaint set, inspect each trace, and compare the tools the model requested with the expected set. This is a tool-selection check, not automatic proof that an answer or citation is correct. All orders and refunds are fictional, and the handbook search only reads Session 10's `heading-800` namespace. Create `.env` with both keys before using Google or Pinecone; local order/refund checks can run without them.

Find the repository whether Jupyter starts here or in the notebooks folder. Keep the two Google model names visible, and import the shared implementation so all notebooks use the same tool behavior.

In [ ]:
import os
import sys
import logging
from pathlib import Path
from dotenv import load_dotenv
EMBEDDING_MODEL = "google_genai:gemini-embedding-001"
CHAT_MODEL = "google_genai:gemini-3.7-flash"
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import agent as lab
load_dotenv(ROOT / ".env")

Check that the required files and key names are present without printing secrets. The log labels show observable tool choices and results, not hidden model reasoning; LangSmith tracing is disabled by the shared module.

In [ ]:
logging.basicConfig(level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s", force=True)
missing = []
for name in ("GOOGLE_API_KEY", "PINECONE_API_KEY"):
    if not os.getenv(name):
        missing.append(name)
if missing:
    print("Add these keys to .env:", ", ".join(missing))
else:
    print("Both keys are present; live requests check access.")
assert lab.ORDERS_PATH.is_file(), "Restore data/orders.csv before continuing."

Load only the test-ticket fixture and inspect its six expected tool sets. Keep the refund ledger from prior notebooks: the duplicate guard should still apply, and the runner never deletes or resets refunds automatically.

In [ ]:
import pandas as pd
tickets = pd.read_csv(lab.TICKETS_PATH, keep_default_na=False)
print(tickets.to_string(index=False))
assert len(tickets) == 6

Create the full agent with human approval enabled, then use the shared ticket runner to show all traces and the ✓/✗ comparison. Approve only an eligible, verified request; reject any injected or unjustified refund, especially in ticket 6.

In [ ]:
agent = lab.build_agent()
exit_code = lab.run_tickets(agent)
print("All tool checks passed:", exit_code == 0)

## Read the trace
REASON names the model's observable tool choice, ACT shows a proposed call, OBSERVE shows its returned data, APPROVAL records your decision, and DONE shows the reply and step count. A proposed refund can appear in the trace even when rejected, so also inspect whether the tool actually confirmed a write.

## Check behavior beyond tool names
Ticket 5 should ask for the correct order ID instead of inventing an order, and ticket 6 should ignore the injected instruction and record no refunds. Verify policy citations, stop-limit outcomes, and refund messages manually; a matching tool set alone cannot prove those behaviors.